# **Word Embeddings**

The requirements for this stage of the analysis are:

1. Most similar words (Using Static Word Embedding) from the top occurring words (max of 3 words, from Term-Frequency Language Model).
2. PCA of the words from the top 3 words (from the Term-Frequency Language Model).
3. Application of a Hugging Face model to your dataset (BERT-based topic model, BERT-based Sentiment / Emotion model)

As such, I am going to apply similarly the processes used in the lecture and course notes on the preprocessed dataset, considering mostly the `cleaned text` attribute of reddit_cleaned.csv

In [46]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer
import joblib
import pandas as pd

from wordcloud import WordCloud
import matplotlib.pyplot as plt

In [47]:
corpus = pd.read_csv("../Binwag - PhilippineRedditAgricultureAnalysis/data/reddit_cleaned.csv")
corpus.head()

,post_title,comment,created_utc,cleaned_comment,cleaned_post_title,cleaned_text
0,"Sen. Marcoleta to hold Blue Ribbon Committee, ...",NaN,2025-07-07 11:40:47,NaN,sen marcoleta hold blue ribbon committee sen b...,sen marcoleta hold blue ribbon committee sen b...
1,"Sen. Marcoleta to hold Blue Ribbon Committee, ...","Buti naman si Kiko papalit sa Agriculture, dam...",2025-07-07 12:38:47,buti naman si kiko papalit sa agriculture dam ...,sen marcoleta hold blue ribbon committee sen b...,sen marcoleta hold blue ribbon committee sen b...
2,"Sen. Marcoleta to hold Blue Ribbon Committee, ...",May sumpa rin ang blue ribbon. \n\nTolentino -...,2025-07-07 12:21:50,may sumpa rin ang blue ribbon tolentino talogo...,sen marcoleta hold blue ribbon committee sen b...,sen marcoleta hold blue ribbon committee sen b...
3,"Sen. Marcoleta to hold Blue Ribbon Committee, ...",Bobo naman bakit sa kanya Blue Ribbon Committee,2025-07-07 11:51:43,bobo naman bakit sa kanya blue ribbon committee,sen marcoleta hold blue ribbon committee sen b...,sen marcoleta hold blue ribbon committee sen b...
4,"Sen. Marcoleta to hold Blue Ribbon Committee, ...",Blue Ribbon Committee is a curse seat lol yeah...,2025-07-07 12:46:06,blue ribbon committee curse seat lol yeah maga...,sen marcoleta hold blue ribbon committee sen b...,sen marcoleta hold blue ribbon committee sen b...


## **Bag of Words Model**

In [48]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

added_stopwords = [
    "sa", "na", "ng", "ang", "mga", "yung", "pa", "lang", "naman", "ba",
    "ko", "mo", "nila", "pero", "daw", "din", "rin", "kasi", "nga", "eh",
    "si", "ni", "kay", "para", "kung", "ay", "lahat", "wala", "may", "meron",
    "dapat", "talaga", "ano", "oo", "hindi", "diba", "sobrang", "super",
    "mas", "isa", "dalawa", "tatlo", "sen", "say", "ma", "make", "ako", "ka"
]

custom_stopwords = list(set(ENGLISH_STOP_WORDS).union(added_stopwords))

In [ ]:
bow_vectorizer = CountVectorizer(
    ngram_range=(1, 2),
    stop_words=custom_stopwords
)

# doc2vec_bow = bow_vectorizer.fit_transform(corpus['cleaned_text'])
# doc2vec_bow = pd.DataFrame(doc2vec_bow.toarray(), columns=bow_vectorizer.get_feature_names_out())
# doc2vec_bow.head()

# Save to file
# doc2vec_bow.to_pickle('doc2vec_bow.pkl')

# # Load from file
# doc2vec_bow = pd.read_pickle('doc2vec_bow.pkl')

# Notebook does not allow .toarray() since it causes memory allocation errors.
doc2vec_bow_sparse = bow_vectorizer.fit_transform(corpus['cleaned_text'])
joblib.dump((doc2vec_bow_sparse, bow_vectorizer), "doc2vec_bow.pkl")
doc2vec_bow_sparse, bow_vectorizer = joblib.load("doc2vec_bow.pkl")

# If you need a small sample as DataFrame
sample_df = pd.DataFrame(
    doc2vec_bow_sparse[:10].toarray(), 
    columns=bow_vectorizer.get_feature_names_out()
)
print(sample_df.head())

   000  000 000  000 170  000 license  00000291  00000291 sys  002  \
0    0        0        0            0         0             0    0   
1    0        0        0            0         0             0    0   
2    0        0        0            0         0             0    0   
3    0        0        0            0         0             0    0   
4    0        0        0            0         0             0    0   

   002 percentage  01  01 percent  ...  zosimo  zosimo tagal  zubiri  \
0               0   0           0  ...       0             0       0   
1               0   0           0  ...       0             0       0   
2               0   0           0  ...       0             0       0   
3               0   0           0  ...       0             0       0   
4               0   0           0  ...       0             0       0   

   zubiri cayetano  zubiri high  zubiri instead  zulueta  zulueta nandyan  \
0                0            0               0        0             

## **TF-IDF Model**

I did not continue doing TF-IDF Model since it runs out of memory to process the array with shape (13171,219130)

In [ ]:
# tfidf_vectorizer = TfidfVectorizer(
#   # unigram
#   # ngram_range=(1, 1),
#   # bigram
#   ngram_range=(1, 2),
#   # trigram
#   # ngram_range=(1, 3),
# )

# doc2vec_tfidf = tfidf_vectorizer.fit_transform(corpus['cleaned_text'])
# doc2vec_tfidf = pd.DataFrame(doc2vec_tfidf.toarray(), columns=tfidf_vectorizer.get_feature_names_out())
# doc2vec_tfidf.head()

# # Save to file
# doc2vec_tfidf.to_pickle('doc2vec_tfidf.pkl')

# # Load from file
# doc2vec_tfidf = pd.read_pickle('doc2vec_tfidf.pkl')

# doc2vec_tfidf.apply(lambda document:
#                     top_n_grams(document, top_n=5),
#                     axis=1)

# # Wordcloud using BoW and TF-IDF
# def plot_wordcloud(data, title):
#   wordcloud = WordCloud(
#     width=800, height=400, background_color='white'
#   ).generate_from_frequencies(data)
#   plt.figure(figsize=(10, 6))
#   plt.imshow(wordcloud, interpolation='bilinear')
#   plt.axis('off')
#   plt.title(title)
#   plt.show()


# tfidf_frequencies = doc2vec_tfidf.sum(axis=0).sort_values(ascending=False)
# tfidf_frequencies.head(5)
# plot_wordcloud(tfidf_frequencies, 'Wordcloud using TF-IDF')

In [ ]:
def top_n_grams(document, top_n=10):
  return document.sort_values(ascending=False).index[:top_n].tolist()

In [ ]:
doc2vec_bow.apply(lambda document:
                  top_n_grams(document, top_n=5),
                  axis=1)

In [ ]:
# Wordcloud using BoW and TF-IDF
def plot_wordcloud(data, title):
  wordcloud = WordCloud(
    width=800, height=400, background_color='white'
  ).generate_from_frequencies(data)
  plt.figure(figsize=(10, 6))
  plt.imshow(wordcloud, interpolation='bilinear')
  plt.axis('off')
  plt.title(title)
  plt.show()

In [ ]:
bow_frequencies = doc2vec_bow.sum(axis=0).sort_values(ascending=False)
print(bow_frequencies.head(5))
plot_wordcloud(bow_frequencies, 'Wordcloud using BoW')